# Trabalho 2 - Recuperação da Informação

Grupo:

- Arthur Trottmann Ramos (14681052)
- Maicon Chaves Marques (14593530)

## Instalação de Dependências

In [1]:
%pip install nltk numpy pandas matplotlib sentence-transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.


## Pré-processamento

A representação lexical usa normalização para minúsculas, tokenização, remoção de stopwords em inglês e stemming Porter. O mesmo pré-processamento será aplicado aos documentos e às consultas.

In [20]:
import json
import math
import re
import string
from collections import Counter, defaultdict
from time import perf_counter
from pathlib import Path

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

try:
    STOPWORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    STOPWORDS = set(stopwords.words('english'))
stemmer = PorterStemmer()


def tokenize(text):
    """Normaliza para minúsculas e retorna tokens sem stemming."""
    return re.findall(r"\b\w+\b", str(text).lower())


def preprocess_tokens(tokens):
    """Remove stopwords e aplica stemming a tokens já normalizados."""
    return [stemmer.stem(token) for token in tokens if token not in STOPWORDS]


def preprocess(text):
    """Tokeniza, normaliza, remove stopwords e aplica stemming Porter."""
    return preprocess_tokens(tokenize(text))


## Leitura do NF-Corpus

Os arquivos JSONL são carregados como dicionários indexados pelo identificador do documento ou da consulta. Os três arquivos de qrels são lidos separadamente e também reunidos em um único DataFrame.

In [21]:
# O notebook deve ser executado a partir da pasta raiz do projeto.
DATA_DIR = Path('nfcorpus/nfcorpus')
EXPERIMENTS_DIR = Path('Experiments')
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)


def read_jsonl(path):
    """Lê um arquivo JSON Lines e retorna registros indexados pelo campo _id."""
    records = {}
    with path.open(encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            record_id = record.get('_id')
            if record_id is None:
                raise ValueError(f'Registro sem _id em {path}, linha {line_number}')
            records[record_id] = record
    return records


documents = read_jsonl(DATA_DIR / 'corpus.jsonl')
queries = read_jsonl(DATA_DIR / 'queries.jsonl')

qrels = {
    split: pd.read_csv(DATA_DIR / 'qrels' / f'{split}.tsv', sep='\t')
    for split in ('train', 'dev', 'test')
}
qrels_all = pd.concat(qrels.values(), ignore_index=True)

print(f'Documentos: {len(documents):,}')
print(f'Consultas: {len(queries):,}')
print(f'Relevâncias (train/dev/test): {len(qrels_all):,}')
print('Colunas dos qrels:', list(qrels_all.columns))


Documentos: 3,633
Consultas: 3,237
Relevâncias (train/dev/test): 134,294
Colunas dos qrels: ['query-id', 'corpus-id', 'score']


## Construção do Índice Invertido

Cada documento é representado pela concatenação de título e texto. O índice associa cada termo processado aos documentos em que ocorre e à frequência do termo em cada documento. Também são armazenados o comprimento processado de cada documento e o total de documentos, usados no cálculo de BM25.

In [15]:
index_build_start = perf_counter()
inverted_index = defaultdict(dict)
document_lengths = {}
raw_vocabulary = set()

for doc_id, document in documents.items():
    title = document.get('title', '')
    text = document.get('text', '')
    raw_tokens = tokenize(f'{title} {text}')
    raw_vocabulary.update(raw_tokens)
    tokens = preprocess_tokens(raw_tokens)
    term_frequencies = Counter(tokens)
    document_lengths[doc_id] = len(tokens)

    for term, frequency in term_frequencies.items():
        inverted_index[term][doc_id] = frequency

# Converte para dict comum para facilitar inspeção e serialização posteriores.
inverted_index = dict(inverted_index)
index_build_time_seconds = perf_counter() - index_build_start
num_documents = len(documents)
avg_document_length = (
    sum(document_lengths.values()) / num_documents if num_documents else 0
)
num_postings = sum(len(postings) for postings in inverted_index.values())

def document_frequency(term):
    """Retorna em quantos documentos o termo processado aparece."""
    return len(inverted_index.get(term, {}))

print(f'Termos no vocabulário: {len(inverted_index):,}')
print(f'Total de postings: {num_postings:,}')
print(f'Tempo de construção do índice: {index_build_time_seconds:.4f} s')
print(f'Tamanho médio dos documentos: {avg_document_length:.2f} tokens')
print('Exemplo de postings:', next(iter(inverted_index.items())))


Termos no vocabulário: 19,054
Total de postings: 360,200
Tempo de construção do índice: 3.2794 s
Tamanho médio dos documentos: 165.82 tokens
Exemplo de postings: ('statin', {'MED-10': 12, 'MED-14': 6, 'MED-870': 1, 'MED-1193': 11, 'MED-1207': 2, 'MED-1258': 2, 'MED-1283': 1, 'MED-1429': 1, 'MED-1486': 3, 'MED-1554': 1, 'MED-1639': 1, 'MED-1874': 1, 'MED-1887': 2, 'MED-2428': 4, 'MED-2429': 8, 'MED-2431': 11, 'MED-2432': 1, 'MED-2525': 1, 'MED-3204': 1, 'MED-3430': 1, 'MED-4559': 9, 'MED-4562': 5, 'MED-4825': 1, 'MED-4827': 1, 'MED-4828': 8, 'MED-4829': 10, 'MED-4830': 8})


## Busca Lexical com BM25

A busca calcula pontuações apenas para documentos encontrados nas posting lists dos termos da consulta. O tempo médio pode ser medido sobre as consultas julgadas de uma partição, sem usar o resultado para ajustar parâmetros no conjunto de teste.

In [22]:
BM25_K1 = 1.2
BM25_B = 0.75


def search_bm25(query, top_k=10, k1=BM25_K1, b=BM25_B):
    """Retorna os top_k documentos ordenados pela pontuação BM25."""
    if k1 <= 0 or not 0 <= b <= 1:
        raise ValueError('BM25 requer k1 > 0 e 0 <= b <= 1')
    if top_k <= 0 or num_documents == 0 or avg_document_length == 0:
        return []

    query_terms = set(preprocess(query))
    scores = defaultdict(float)

    for term in query_terms:
        postings = inverted_index.get(term)
        if not postings:
            continue

        df = len(postings)
        # IDF BM25 com suavização, que mantém valores positivos.
        idf = math.log1p((num_documents - df + 0.5) / (df + 0.5))

        for doc_id, term_frequency in postings.items():
            doc_length = document_lengths[doc_id]
            length_norm = k1 * (1 - b + b * doc_length / avg_document_length)
            tf_weight = (term_frequency * (k1 + 1)) / (term_frequency + length_norm)
            scores[doc_id] += idf * tf_weight

    ranked = sorted(scores.items(), key=lambda item: (-item[1], item[0]))[:top_k]
    return [
        {'doc_id': doc_id, 'score': float(score)}
        for doc_id, score in ranked
    ]


def measure_bm25_query_time(split='dev', top_k=100, repetitions=1):
    """Mede a média por consulta para IDs julgados na partição indicada."""
    if split not in qrels:
        raise ValueError(f'Partição desconhecida: {split}. Use train, dev ou test.')
    if repetitions <= 0:
        raise ValueError('repetitions deve ser maior que zero')

    query_ids = sorted(set(qrels[split]['query-id']) & set(queries))
    if not query_ids:
        raise ValueError(f'Nenhuma consulta de {split} foi encontrada em queries.jsonl')

    start = perf_counter()
    for _ in range(repetitions):
        for query_id in query_ids:
            search_bm25(queries[query_id].get('text', ''), top_k=top_k)
    elapsed = perf_counter() - start
    total_queries = len(query_ids) * repetitions
    return {
        'split': split,
        'num_queries': len(query_ids),
        'repetitions': repetitions,
        'average_query_time_seconds': elapsed / total_queries,
    }


# Exemplo de uso após criar o índice:
# search_bm25(queries['PLAIN-3']['text'], top_k=10)
# Depois de fixar os parâmetros com dev, pode-se medir no conjunto final:
# measure_bm25_query_time(split='test', top_k=100)


## Tolerância a erros: k-gramas, Jaccard e distância de edição

O vocabulário de correção é formado pelos tokens do corpus antes de stemming, conforme o item 4 do PDF. Termos conhecidos são preservados; para termos fora do vocabulário, k-gramas geram candidatos, Jaccard filtra candidatos e distância de Levenshtein seleciona a alternativa mais próxima. O tamanho do k-grama e os limiares serão ajustados em `dev`, antes de aplicar a configuração às consultas de `test`.


In [ ]:
KGRAM_SIZE = 3
JACCARD_THRESHOLD = 0.30
MAX_EDIT_DISTANCE = 2
TYPO_OPERATIONS = ('insertion', 'deletion', 'substitution', 'transposition')
TYPO_LABELS = {
    'insertion': 'inserção',
    'deletion': 'remoção',
    'substitution': 'substituição',
    'transposition': 'troca de ordem',
}


def character_kgrams(term, k=KGRAM_SIZE):
    """Retorna k-gramas de caracteres com marcadores de início e fim."""
    if k <= 0:
        raise ValueError('k deve ser maior que zero')
    padded = '$' * (k - 1) + term + '$' * (k - 1)
    if len(padded) < k:
        return {padded}
    return {padded[i:i + k] for i in range(len(padded) - k + 1)}


def build_kgram_index(vocabulary, k=KGRAM_SIZE):
    """Cria índice invertido de k-gramas sobre o vocabulário recebido."""
    index = defaultdict(set)
    for vocabulary_term in vocabulary:
        for gram in character_kgrams(vocabulary_term, k):
            index[gram].add(vocabulary_term)
    return dict(index)


def jaccard_coefficient(left, right):
    """Calcula J(A,B) = |A ∩ B| / |A ∪ B|."""
    union = left | right
    return len(left & right) / len(union) if union else 1.0


def edit_distance(left, right):
    """Distância de Levenshtein usando inserção, remoção e substituição."""
    if len(left) < len(right):
        left, right = right, left
    previous = list(range(len(right) + 1))
    for i, left_char in enumerate(left, start=1):
        current = [i]
        for j, right_char in enumerate(right, start=1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + (left_char != right_char),
            ))
        previous = current
    return previous[-1]


# Índice auxiliar de k-gramas do vocabulário não stemmizado.
kgram_index_build_start = perf_counter()
kgram_index = build_kgram_index(raw_vocabulary, KGRAM_SIZE)
kgram_index_build_time_seconds = perf_counter() - kgram_index_build_start
print(f'Termos no vocabulário de correção (antes do stemming): {len(raw_vocabulary):,}')
print(f'Tempo de construção do índice de k-gramas: {kgram_index_build_time_seconds:.4f} s')


def best_spelling_candidate(term, jaccard_threshold=JACCARD_THRESHOLD,
                            max_distance=MAX_EDIT_DISTANCE, k=None,
                            candidate_index=None):
    """Encontra correção OOV: candidatos por k-gramas, Jaccard e Levenshtein."""
    if term in raw_vocabulary:
        return None
    if k is None:
        k = KGRAM_SIZE

    index = kgram_index if candidate_index is None else candidate_index
    query_grams = character_kgrams(term, k)
    candidates = set()
    for gram in query_grams:
        candidates.update(index.get(gram, set()))

    filtered = []
    for candidate in candidates:
        similarity = jaccard_coefficient(query_grams, character_kgrams(candidate, k))
        if similarity < jaccard_threshold:
            continue
        distance = edit_distance(term, candidate)
        if distance <= max_distance:
            filtered.append((distance, -similarity, candidate, similarity))

    if not filtered:
        return None
    distance, _, candidate, similarity = min(filtered)
    return {'term': candidate, 'distance': distance, 'jaccard': similarity}


def correct_query(query, jaccard_threshold=JACCARD_THRESHOLD,
                  max_distance=MAX_EDIT_DISTANCE, k=None,
                  candidate_index=None):
    """Corrige apenas tokens desconhecidos e retorna texto e detalhes."""
    corrections = []
    if k is None:
        k = KGRAM_SIZE

    def replace_unknown(match):
        observed = match.group(0).lower()
        choice = best_spelling_candidate(
            observed,
            jaccard_threshold=jaccard_threshold,
            max_distance=max_distance,
            k=k,
            candidate_index=candidate_index,
        )
        if choice is None:
            return match.group(0)
        corrections.append({
            'observed': observed,
            'replacement': choice['term'],
            'edit_distance': choice['distance'],
            'jaccard': choice['jaccard'],
        })
        return choice['term']

    corrected = re.sub(r'\b\w+\b', replace_unknown, str(query))
    return corrected, corrections


def _typo_candidates(term, operation):
    """Gera corrupções de um único tipo, em ordem determinística."""
    alphabet = string.ascii_lowercase
    if operation == 'insertion':
        return [term[:i] + char + term[i:]
                for i in range(len(term) + 1) for char in alphabet]
    if operation == 'deletion':
        return [term[:i] + term[i + 1:]
                for i in range(len(term)) if len(term) > 2]
    if operation == 'substitution':
        return [term[:i] + char + term[i + 1:]
                for i in range(len(term)) for char in alphabet if char != term[i]]
    if operation == 'transposition':
        return [term[:i] + term[i + 1] + term[i] + term[i + 2:]
                for i in range(len(term) - 1) if term[i] != term[i + 1]]
    raise ValueError(f'Tipo de erro desconhecido: {operation}')


def _replace_first_token(query, original, replacement):
    pattern = rf'\b{re.escape(original)}\b'
    return re.sub(pattern, replacement, query, count=1, flags=re.IGNORECASE)


def make_oov_typo(query, operation):
    """Introduz um único erro do tipo pedido, sem depender do corretor."""
    query_terms = sorted(set(tokenize(query)), key=lambda term: (-len(term), term))
    for original in query_terms:
        if original not in raw_vocabulary:
            continue
        for typo in _typo_candidates(original, operation):
            if typo in raw_vocabulary:
                continue
            typo_query = _replace_first_token(query, original, typo)
            if typo_query != query:
                return {
                    'query': typo_query,
                    'original_term': original,
                    'typo_term': typo,
                }
    return None


## Experimento de tolerância a erros (item 4 do PDF)

O conjunto fixo de parâmetros será ajustado somente em `dev`, usando consultas e erros gerados sem depender da configuração avaliada. A configuração escolhida será então congelada para o experimento com exatamente 10 consultas de `test`, NDCG@10 original maior que zero e corrupções OOV. Serão comparadas as versões original, com erro e corrigida; os demais experimentos continuam usando as consultas originais.


In [ ]:
def _qrels_by_query(qrels_frame):
    judgments = defaultdict(dict)
    for _, row in qrels_frame.iterrows():
        qid = row['query-id']
        doc_id = row['corpus-id']
        judgments[qid][doc_id] = max(
            float(row['score']), judgments[qid].get(doc_id, float('-inf'))
        )
    return dict(judgments)


def ndcg_at_k(ranked_doc_ids, relevant_grades, k=10):
    """NDCG com ganho linear e IDCG calculado sobre todos os qrels da consulta."""
    if k <= 0:
        return 0.0
    dcg = sum(
        relevant_grades.get(doc_id, 0.0) / math.log2(rank + 2)
        for rank, doc_id in enumerate(ranked_doc_ids[:k])
    )
    ideal_grades = sorted(relevant_grades.values(), reverse=True)[:k]
    idcg = sum(grade / math.log2(rank + 2)
               for rank, grade in enumerate(ideal_grades))
    return dcg / idcg if idcg > 0 else 0.0


def _bm25_ndcg(query_text, query_id, judgments, k=10):
    results = search_bm25(query_text, top_k=k)
    return ndcg_at_k(
        [result['doc_id'] for result in results],
        judgments.get(query_id, {}),
        k=k,
    )


def prepare_typo_cases(n_queries=10, split='dev', k=10):
    """Prepara um conjunto fixo de consultas/erros, independente do corretor."""
    if split not in qrels:
        raise ValueError(f'Partição desconhecida: {split}. Use train, dev ou test.')
    judgments = _qrels_by_query(qrels[split])
    query_ids = sorted(set(judgments) & set(queries))
    cases = []
    for query_id in query_ids:
        if len(cases) == n_queries:
            break
        original_query = queries[query_id].get('text', '')
        original_ndcg = _bm25_ndcg(original_query, query_id, judgments, k=k)
        if original_ndcg <= 0:
            continue
        operation = TYPO_OPERATIONS[len(cases) % len(TYPO_OPERATIONS)]
        typo = make_oov_typo(original_query, operation)
        if typo is None:
            continue
        error_ndcg = _bm25_ndcg(typo['query'], query_id, judgments, k=k)
        cases.append({
            'query_id': query_id,
            'query_original': original_query,
            'query_erro': typo['query'],
            'tipo_erro': operation,
            'token_original': typo['original_term'],
            'token_com_erro': typo['typo_term'],
            'NDCG10_original': original_ndcg,
            'NDCG10_erro': error_ndcg,
        })
    if len(cases) != n_queries:
        raise RuntimeError(
            f'Foram encontradas apenas {len(cases)} consultas elegíveis em {split}; '
            f'eram necessárias {n_queries}.'
        )
    return cases


def tune_typo_parameters(n_queries=10, k=10,
                         kgram_sizes=(2, 3, 4),
                         jaccard_thresholds=(0.20, 0.30, 0.40, 0.50),
                         max_edit_distances=(1, 2)):
    """Escolhe parâmetros exclusivamente em dev com os mesmos erros em cada configuração."""
    dev_cases = prepare_typo_cases(n_queries=n_queries, split='dev', k=k)
    dev_judgments = _qrels_by_query(qrels['dev'])
    results = []

    for gram_size in kgram_sizes:
        candidate_index = build_kgram_index(raw_vocabulary, gram_size)
        for threshold in jaccard_thresholds:
            for max_distance in max_edit_distances:
                recoveries = []
                target_hits = 0
                corrected_scores = []
                damaged_cases = 0

                for case in dev_cases:
                    corrected_query, corrections = correct_query(
                        case['query_erro'],
                        jaccard_threshold=threshold,
                        max_distance=max_distance,
                        k=gram_size,
                        candidate_index=candidate_index,
                    )
                    corrected_ndcg = _bm25_ndcg(
                        corrected_query, case['query_id'],
                        _qrels_by_query(qrels['dev']), k=k,
                    )
                    corrected_scores.append(corrected_ndcg)
                    target_hit = any(
                        item['observed'] == case['token_com_erro']
                        and item['replacement'] == case['token_original']
                        for item in corrections
                    )
                    target_hits += int(target_hit)
                    loss = case['NDCG10_original'] - case['NDCG10_erro']
                    if loss > 0:
                        damaged_cases += 1
                        recoveries.append((corrected_ndcg - case['NDCG10_erro']) / loss)

                results.append({
                    'kgram_size': gram_size,
                    'jaccard_threshold': threshold,
                    'max_edit_distance': max_distance,
                    'mean_recovery_fraction': sum(recoveries) / len(recoveries) if recoveries else 0.0,
                    'target_correction_accuracy': target_hits / len(dev_cases),
                    'mean_corrected_ndcg10': sum(corrected_scores) / len(corrected_scores),
                    'num_dev_cases': len(dev_cases),
                    'num_cases_with_performance_loss': damaged_cases,
                })

    # Critério: acerto do token artificial, NDCG corrigido e fração recuperada;
    # desempate favorece trigramas e configurações mais conservadoras.
    results.sort(key=lambda row: (
        -row['target_correction_accuracy'],
        -row['mean_corrected_ndcg10'],
        -row['mean_recovery_fraction'],
        abs(row['kgram_size'] - 3),
        -row['jaccard_threshold'],
        row['max_edit_distance'],
    ))
    return pd.DataFrame(results), results[0]


def run_typo_experiment(n_queries=10, split='test', k=10,
                        jaccard_threshold=JACCARD_THRESHOLD,
                        max_distance=MAX_EDIT_DISTANCE,
                        kgram_size=KGRAM_SIZE):
    """Roda original/erro/corrigida no BM25 em uma partição."""
    if split not in qrels:
        raise ValueError(f'Partição desconhecida: {split}. Use train, dev ou test.')
    if split == 'test' and n_queries != 10:
        raise ValueError('O protocolo do PDF exige exatamente 10 consultas de test.')
    if n_queries <= 0:
        raise ValueError('n_queries deve ser maior que zero')

    judgments = _qrels_by_query(qrels[split])
    cases = prepare_typo_cases(n_queries=n_queries, split=split, k=k)
    records = []
    candidate_index = build_kgram_index(raw_vocabulary, kgram_size)

    for case in cases:
        error_ndcg = case['NDCG10_erro']
        corrected_query, corrections = correct_query(
            case['query_erro'],
            jaccard_threshold=jaccard_threshold,
            max_distance=max_distance,
            k=kgram_size,
            candidate_index=candidate_index,
        )
        corrected_ndcg = _bm25_ndcg(
            corrected_query, case['query_id'], judgments, k=k
        )
        denominator = case['NDCG10_original'] - error_ndcg
        if denominator > 0:
            recovered_fraction = (corrected_ndcg - error_ndcg) / denominator
            recovery_status = 'calculada'
        else:
            recovered_fraction = None
            recovery_status = 'erro não reduziu o desempenho'

        target_corrections = [
            item for item in corrections
            if item['observed'] == case['token_com_erro']
        ]
        records.append({
            'query_id': case['query_id'],
            'query_original': case['query_original'],
            'query_erro': case['query_erro'],
            'query_corrigida': corrected_query,
            'tipo_erro': TYPO_LABELS[case['tipo_erro']],
            'token_original': case['token_original'],
            'token_com_erro': case['token_com_erro'],
            'token_corrigido': target_corrections[0]['replacement'] if target_corrections else '',
            'correcoes_aplicadas': json.dumps(corrections, ensure_ascii=False),
            'NDCG10_original': case['NDCG10_original'],
            'NDCG10_erro': error_ndcg,
            'NDCG10_corrigida': corrected_ndcg,
            'fracao_desempenho_recuperada': recovered_fraction,
            'status_recuperacao': recovery_status,
        })
    return pd.DataFrame(records)


# Primeiro ajusta somente em dev; depois congela os parâmetros e executa test.
dev_tuning_results, best_typo_parameters = tune_typo_parameters(n_queries=10, k=10)
JACCARD_THRESHOLD = best_typo_parameters['jaccard_threshold']
MAX_EDIT_DISTANCE = best_typo_parameters['max_edit_distance']
KGRAM_SIZE = best_typo_parameters['kgram_size']
kgram_index = build_kgram_index(raw_vocabulary, KGRAM_SIZE)
print('Melhor configuração escolhida em dev:', best_typo_parameters)
dev_tuning_results.to_csv(EXPERIMENTS_DIR / 'typo_dev_tuning.csv', index=False)
display(dev_tuning_results.head(10))

typo_results = run_typo_experiment(
    n_queries=10,
    split='test',
    k=10,
    jaccard_threshold=JACCARD_THRESHOLD,
    max_distance=MAX_EDIT_DISTANCE,
    kgram_size=KGRAM_SIZE,
)
typo_results.to_csv(EXPERIMENTS_DIR / 'typo_experiment.csv', index=False)
display(typo_results)


## Recuperação semântica com sentence-BERT

Os documentos são codificados uma única vez com o bi-encoder pré-treinado `sentence-transformers/all-MiniLM-L6-v2`. Os embeddings normalizados e o alinhamento entre vetor e ID são persistidos em `Experiments/semantic/`. O cache só é reutilizado quando modelo, configuração e hash do texto da coleção conferem. Consultas usam o texto original (sem stemming lexical) e são ranqueadas por cosseno, calculado como produto interno entre vetores normalizados. A busca percorre todos os embeddings da coleção, adequada ao tamanho do NF-Corpus.

A primeira execução precisa baixar os pesos do modelo se eles não estiverem no cache local. O comprimento máximo efetivo e a quantidade de documentos truncados são medidos com o tokenizador do próprio modelo.

In [8]:
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import sentence_transformers
import torch
from sentence_transformers import SentenceTransformer

SEMANTIC_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
SEMANTIC_CACHE_DIR = EXPERIMENTS_DIR / 'semantic'
SEMANTIC_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SEMANTIC_EMBEDDINGS_PATH = SEMANTIC_CACHE_DIR / 'all_minilm_l6_v2_document_embeddings.npy'
SEMANTIC_DOC_IDS_PATH = SEMANTIC_CACHE_DIR / 'all_minilm_l6_v2_document_ids.json'
SEMANTIC_MANIFEST_PATH = SEMANTIC_CACHE_DIR / 'all_minilm_l6_v2_manifest.json'
SEMANTIC_NORMALIZE_EMBEDDINGS = True
SEMANTIC_BATCH_SIZE = 32

# Texto sem stemming/remoção de stopwords: preserva a entrada natural do sentence-BERT.
semantic_document_ids = sorted(documents)
semantic_document_texts = [
    f"{documents[doc_id].get('title', '') or ''} {documents[doc_id].get('text', '') or ''}".strip()
    for doc_id in semantic_document_ids
]
corpus_hasher = hashlib.sha256()
for doc_id, text in zip(semantic_document_ids, semantic_document_texts):
    corpus_hasher.update(doc_id.encode('utf-8'))
    corpus_hasher.update(b'\0')
    corpus_hasher.update(text.encode('utf-8'))
    corpus_hasher.update(b'\n')
semantic_corpus_sha256 = corpus_hasher.hexdigest()
semantic_device = 'cuda' if torch.cuda.is_available() else 'cpu'
semantic_model = SentenceTransformer(SEMANTIC_MODEL_NAME, device=semantic_device)
semantic_max_seq_length = int(semantic_model.max_seq_length)

# Mede truncamento com o mesmo tokenizador e limite usado pelo encoder.
semantic_token_lengths = []
for start in range(0, len(semantic_document_texts), SEMANTIC_BATCH_SIZE):
    batch = semantic_document_texts[start:start + SEMANTIC_BATCH_SIZE]
    encoded = semantic_model.tokenizer(
        batch, add_special_tokens=True, truncation=False, padding=False
    )
    semantic_token_lengths.extend(len(token_ids) for token_ids in encoded['input_ids'])
semantic_truncated_documents = sum(
    length > semantic_max_seq_length for length in semantic_token_lengths
)

semantic_cache_valid = False
if all(path.exists() for path in (SEMANTIC_EMBEDDINGS_PATH, SEMANTIC_DOC_IDS_PATH, SEMANTIC_MANIFEST_PATH)):
    with SEMANTIC_MANIFEST_PATH.open(encoding='utf-8') as file:
        semantic_cached_manifest = json.load(file)
    with SEMANTIC_DOC_IDS_PATH.open(encoding='utf-8') as file:
        semantic_cached_ids = json.load(file)
    semantic_cached_embeddings = np.load(SEMANTIC_EMBEDDINGS_PATH, mmap_mode='r')
    semantic_cache_valid = (
        semantic_cached_manifest.get('model_name') == SEMANTIC_MODEL_NAME
        and semantic_cached_manifest.get('corpus_sha256') == semantic_corpus_sha256
        and semantic_cached_manifest.get('normalized_embeddings') is SEMANTIC_NORMALIZE_EMBEDDINGS
        and semantic_cached_manifest.get('max_seq_length_tokens') == semantic_max_seq_length
        and semantic_cached_ids == semantic_document_ids
        and semantic_cached_embeddings.ndim == 2
        and semantic_cached_embeddings.shape[0] == len(semantic_document_ids)
        and semantic_cached_embeddings.shape[1] == semantic_model.get_embedding_dimension()
    )

if semantic_cache_valid:
    semantic_document_embeddings = np.load(SEMANTIC_EMBEDDINGS_PATH)
    semantic_embedding_build_seconds = semantic_cached_manifest.get('embedding_build_seconds')
    semantic_cache_status = 'carregado do cache validado'
else:
    semantic_embedding_start = time.perf_counter()
    semantic_document_embeddings = semantic_model.encode(
        semantic_document_texts,
        batch_size=SEMANTIC_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=SEMANTIC_NORMALIZE_EMBEDDINGS,
    ).astype(np.float32, copy=False)
    semantic_embedding_build_seconds = time.perf_counter() - semantic_embedding_start
    np.save(SEMANTIC_EMBEDDINGS_PATH, semantic_document_embeddings)
    with SEMANTIC_DOC_IDS_PATH.open('w', encoding='utf-8') as file:
        json.dump(semantic_document_ids, file, ensure_ascii=False)
    semantic_cached_manifest = {
        'model_name': SEMANTIC_MODEL_NAME,
        'corpus_sha256': semantic_corpus_sha256,
        'document_count': len(semantic_document_ids),
        'embedding_dimension': int(semantic_document_embeddings.shape[1]),
        'normalized_embeddings': SEMANTIC_NORMALIZE_EMBEDDINGS,
        'similarity': 'cosine (dot product of L2-normalized vectors)',
        'max_seq_length_tokens': semantic_max_seq_length,
        'truncated_document_count': int(semantic_truncated_documents),
        'embedding_build_seconds': semantic_embedding_build_seconds,
        'device': semantic_device,
        'sentence_transformers_version': sentence_transformers.__version__,
        'torch_version': torch.__version__,
        'embedding_dtype': str(semantic_document_embeddings.dtype),
    }
    with SEMANTIC_MANIFEST_PATH.open('w', encoding='utf-8') as file:
        json.dump(semantic_cached_manifest, file, ensure_ascii=False, indent=2)
    semantic_cache_status = 'gerado e salvo'

# Mapeamento explícito do ID para a linha evita desalinhamento ao carregar o cache.
semantic_doc_id_to_row = {doc_id: row for row, doc_id in enumerate(semantic_document_ids)}

def search_semantic(query, top_k=10):
    """Busca exaustiva por cosseno nos embeddings pré-calculados do corpus."""
    if top_k <= 0 or not semantic_document_ids:
        return []
    query_embedding = semantic_model.encode(
        [str(query)], convert_to_numpy=True, normalize_embeddings=True,
    )[0].astype(np.float32, copy=False)
    scores = semantic_document_embeddings @ query_embedding
    best_rows = np.argsort(-scores, kind='stable')[:top_k]
    return [
        {'doc_id': semantic_document_ids[row], 'score': float(scores[row])}
        for row in best_rows
    ]

def measure_semantic_query_time(split='dev', top_k=100, repetitions=1):
    """Mede encode da consulta + busca, sem incluir carga do modelo."""
    if split not in qrels:
        raise ValueError(f'Partição desconhecida: {split}. Use train, dev ou test.')
    if repetitions <= 0:
        raise ValueError('repetitions deve ser maior que zero')
    query_ids = sorted(set(qrels[split]['query-id']) & set(queries))
    if not query_ids:
        return 0.0
    elapsed = 0.0
    for _ in range(repetitions):
        for query_id in query_ids:
            query_text = queries[query_id].get('text', '')
            start = time.perf_counter()
            search_semantic(query_text, top_k=top_k)
            elapsed += time.perf_counter() - start
    return elapsed / (len(query_ids) * repetitions)

print(f'Modelo: {SEMANTIC_MODEL_NAME}')
print(f'Dispositivo: {semantic_device}; sentence-transformers {sentence_transformers.__version__}')
print(f'Documentos: {len(semantic_document_ids):,}; dimensão: {semantic_document_embeddings.shape[1]}')
print(f'Limite: {semantic_max_seq_length} tokens; documentos truncados: {semantic_truncated_documents:,}')
print(f'Embeddings normalizados: {SEMANTIC_NORMALIZE_EMBEDDINGS}; similaridade: cosseno')
print(f'Estado do cache: {semantic_cache_status}')
if semantic_embedding_build_seconds is not None:
    print(f'Tempo registrado para geração dos embeddings: {semantic_embedding_build_seconds:.2f} s')
print(f'Arquivos: {SEMANTIC_EMBEDDINGS_PATH}, {SEMANTIC_DOC_IDS_PATH}, {SEMANTIC_MANIFEST_PATH}')
print('Exemplo de busca:', search_semantic(queries[next(iter(queries))].get('text', ''), top_k=5))
semantic_dev_query_time_seconds = measure_semantic_query_time(split='dev', top_k=100)
print(f'Tempo médio de consulta em dev (encode + busca top-100): {semantic_dev_query_time_seconds:.4f} s/consulta')


Modelo: sentence-transformers/all-MiniLM-L6-v2
Dispositivo: cpu; sentence-transformers 6.1.0
Documentos: 3,633; dimensão: 384
Limite: 256 tokens; documentos truncados: 2,864
Embeddings normalizados: True; similaridade: cosseno
Estado do cache: carregado do cache validado
Tempo registrado para geração dos embeddings: 112.90 s
Arquivos: Experiments/semantic/all_minilm_l6_v2_document_embeddings.npy, Experiments/semantic/all_minilm_l6_v2_document_ids.json, Experiments/semantic/all_minilm_l6_v2_manifest.json
Exemplo de busca: [{'doc_id': 'MED-2439', 'score': 0.6983507871627808}, {'doc_id': 'MED-2434', 'score': 0.648932695388794}, {'doc_id': 'MED-2440', 'score': 0.6331393718719482}, {'doc_id': 'MED-2427', 'score': 0.6134428977966309}, {'doc_id': 'MED-2102', 'score': 0.542138397693634}]
Tempo médio de consulta em dev (encode + busca top-100): 0.0928 s/consulta


## Avaliação lexical × semântica (itens 6 e 7 do PDF)

Com as configurações congeladas, esta etapa avalia BM25 e `all-MiniLM-L6-v2` sobre o mesmo conjunto de consultas julgadas de `test`. Precision@10, Recall@10 e AP@10 tratam todo grau qrel maior que zero como relevante; os denominadores de Recall@10 e AP@10 incluem todos os documentos relevantes nos qrels da consulta. NDCG@10 preserva os graus dos qrels, usa ganho linear e calcula o ideal com todos os julgamentos. São salvos rankings Top100, métricas agregadas e métricas por consulta em CSVs independentes do output do notebook.

A comparação por consulta usa ΔNDCG@10 = NDCG@10 semântico − NDCG@10 lexical. São exportadas as três maiores vantagens de cada método, com texto da consulta, Top5 de ambos, scores, julgamentos, textos documentais e termos processados da consulta encontrados em cada documento como evidência para análise. Os scores de BM25 e cosseno têm escalas diferentes e não são comparados diretamente.

In [9]:
from pathlib import Path
from time import perf_counter
import json
import math

import pandas as pd

EVALUATION_SPLIT = 'test'
EVALUATION_K = 10
EVALUATION_RANKING_DEPTH = 100
EVALUATION_METHODS = ('lexical_bm25', 'semantic_all-MiniLM-L6-v2')


def _evaluation_judgments(qrels_frame):
    """Consolida qrels por consulta, preservando o maior grau por documento."""
    judgments = {}
    for _, row in qrels_frame.iterrows():
        query_id = row['query-id']
        doc_id = row['corpus-id']
        grade = float(row['score'])
        per_query = judgments.setdefault(query_id, {})
        per_query[doc_id] = max(grade, per_query.get(doc_id, float('-inf')))
    return judgments


def _retrieval_metrics(ranked_doc_ids, relevance_grades, k=10):
    """Retorna P@k, R@k, AP@k e NDCG@k conforme os itens 6–7 do PDF."""
    relevant_ids = {doc_id for doc_id, grade in relevance_grades.items() if grade > 0}
    retrieved = ranked_doc_ids[:k]
    relevant_hits = [doc_id in relevant_ids for doc_id in retrieved]
    hit_count = sum(relevant_hits)
    precision = hit_count / k if k else 0.0
    recall = hit_count / len(relevant_ids) if relevant_ids else 0.0

    precision_sum = 0.0
    seen_relevant = 0
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant_ids:
            seen_relevant += 1
            precision_sum += seen_relevant / rank
    average_precision = precision_sum / len(relevant_ids) if relevant_ids else 0.0

    dcg = sum(
        relevance_grades.get(doc_id, 0.0) / math.log2(rank + 1)
        for rank, doc_id in enumerate(retrieved, start=1)
    )
    ideal_grades = sorted(relevance_grades.values(), reverse=True)[:k]
    idcg = sum(
        grade / math.log2(rank + 1)
        for rank, grade in enumerate(ideal_grades, start=1)
    )
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return {'P10': precision, 'R10': recall, 'AP10': average_precision, 'NDCG10': ndcg}


if EVALUATION_SPLIT not in qrels:
    raise ValueError(f'Partição inexistente: {EVALUATION_SPLIT}')
evaluation_judgments = _evaluation_judgments(qrels[EVALUATION_SPLIT])
evaluation_query_ids = sorted(set(evaluation_judgments) & set(queries))
ordered_document_ids_for_ties = sorted(documents)
if not evaluation_query_ids:
    raise ValueError('Não há consultas em comum entre qrels e queries.jsonl')

ranking_records = {'lexical_bm25': [], 'semantic_all-MiniLM-L6-v2': []}
query_metric_records = []
query_time_totals = {method: 0.0 for method in EVALUATION_METHODS}
rankings_by_query = {}

for query_id in evaluation_query_ids:
    query_text = str(queries[query_id].get('text', '') or '')
    grades = evaluation_judgments[query_id]
    rankings_by_query[query_id] = {}
    for method in EVALUATION_METHODS:
        start = perf_counter()
        if method == 'lexical_bm25':
            ranked = search_bm25(
                query_text, top_k=EVALUATION_RANKING_DEPTH,
                k1=BM25_K1, b=BM25_B,
            )
        else:
            ranked = search_semantic(query_text, top_k=EVALUATION_RANKING_DEPTH)
        # BM25 pontua apenas candidatos de postings. Para completar o CSV Top100,
        # documentos sem correspondência recebem score zero e desempate por doc_id;
        # só são inspecionados IDs suficientes para completar a profundidade pedida.
        if method == 'lexical_bm25' and len(ranked) < EVALUATION_RANKING_DEPTH:
            present_doc_ids = {item['doc_id'] for item in ranked}
            for doc_id in ordered_document_ids_for_ties:
                if doc_id not in present_doc_ids:
                    ranked.append({'doc_id': doc_id, 'score': 0.0})
                    if len(ranked) == EVALUATION_RANKING_DEPTH:
                        break
        elapsed = perf_counter() - start
        query_time_totals[method] += elapsed
        rankings_by_query[query_id][method] = ranked
        ranked_doc_ids = [item['doc_id'] for item in ranked]
        query_metrics = _retrieval_metrics(ranked_doc_ids, grades, k=EVALUATION_K)
        query_metric_records.append({
            'query_id': query_id,
            'model': method,
            **query_metrics,
        })
        for rank, item in enumerate(ranked, start=1):
            ranking_records[method].append({
                'query_id': query_id,
                'doc_id': item['doc_id'],
                'rank': rank,
                'score': float(item['score']),
                'relevance': float(grades.get(item['doc_id'], 0.0)),
            })

metrics_per_query = pd.DataFrame(query_metric_records)
metrics_per_query.to_csv(EXPERIMENTS_DIR / 'metrics_per_query.csv', index=False)

rankings_path = {
    'lexical_bm25': EXPERIMENTS_DIR / 'results_lexical.csv',
    'semantic_all-MiniLM-L6-v2': EXPERIMENTS_DIR / 'results_semantic.csv',
}
for method, records in ranking_records.items():
    ranking_frame = pd.DataFrame(
        records, columns=['query_id', 'doc_id', 'rank', 'score', 'relevance']
    )
    ranking_frame.to_csv(rankings_path[method], index=False)

summary_records = []
for method in EVALUATION_METHODS:
    frame = metrics_per_query.loc[metrics_per_query['model'] == method]
    summary_records.append({
        'split': EVALUATION_SPLIT,
        'model': method,
        'query_count': len(frame),
        'P10': frame['P10'].mean(),
        'R10': frame['R10'].mean(),
        'MAP10': frame['AP10'].mean(),
        'NDCG10': frame['NDCG10'].mean(),
        'average_query_time_seconds': query_time_totals[method] / len(frame),
        'ranking_depth': EVALUATION_RANKING_DEPTH,
        'evaluation_cutoff': EVALUATION_K,
    })
metrics_summary = pd.DataFrame(summary_records)
metrics_summary.to_csv(EXPERIMENTS_DIR / 'metrics_summary.csv', index=False)

print(f'Partição: {EVALUATION_SPLIT}; consultas avaliadas por método: {len(evaluation_query_ids)}')
print('Configuração lexical congelada:', {'model': 'BM25', 'k1': BM25_K1, 'b': BM25_B})
print('Configuração semântica congelada:', {
    'model': SEMANTIC_MODEL_NAME,
    'dimension': int(semantic_document_embeddings.shape[1]),
    'similarity': 'cosine',
})
print(metrics_summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print('Rankings salvos:', ', '.join(str(path) for path in rankings_path.values()))
print(f"Métricas por consulta salvas: {EXPERIMENTS_DIR / 'metrics_per_query.csv'}")
print('Rankings lexicais completados com score zero e desempate por doc_id somente quando faltavam candidatos BM25.')
if any(len(rankings_by_query[qid]['lexical_bm25']) != EVALUATION_RANKING_DEPTH for qid in evaluation_query_ids):
    raise AssertionError('O ranking lexical não contém 100 documentos para todas as consultas.')
if any(len(rankings_by_query[qid]['semantic_all-MiniLM-L6-v2']) != EVALUATION_RANKING_DEPTH for qid in evaluation_query_ids):
    raise AssertionError('O ranking semântico não contém 100 documentos para todas as consultas.')

Partição: test; consultas avaliadas por método: 323
Configuração lexical congelada: {'model': 'BM25', 'k1': 1.2, 'b': 0.75}
Configuração semântica congelada: {'model': 'sentence-transformers/all-MiniLM-L6-v2', 'dimension': 384, 'similarity': 'cosine'}
split                     model  query_count    P10    R10  MAP10  NDCG10  average_query_time_seconds  ranking_depth  evaluation_cutoff
 test              lexical_bm25          323 0.2390 0.1532 0.1218  0.3278                      0.0006            100                 10
 test semantic_all-MiniLM-L6-v2          323 0.2433 0.1550 0.1108  0.3167                      0.0617            100                 10
Rankings salvos: Experiments/results_lexical.csv, Experiments/results_semantic.csv
Métricas por consulta salvas: Experiments/metrics_per_query.csv
Rankings lexicais completados com score zero e desempate por doc_id somente quando faltavam candidatos BM25.


### Comparação por consulta e casos representativos

A célula seguinte seleciona os três maiores ganhos de NDCG@10 de cada sistema e salva o Top5 de cada método com julgamentos e evidência lexical observada.

In [10]:
# Três maiores ganhos para cada sistema, medidos por diferença de NDCG@10.
metric_wide = metrics_per_query.pivot(index='query_id', columns='model', values='NDCG10')
metric_wide['delta_ndcg10_semantic_minus_lexical'] = (
    metric_wide['semantic_all-MiniLM-L6-v2'] - metric_wide['lexical_bm25']
)
semantic_wins = metric_wide.loc[
    metric_wide['delta_ndcg10_semantic_minus_lexical'] > 0
].sort_values(['delta_ndcg10_semantic_minus_lexical'], ascending=False).head(3)
lexical_wins = metric_wide.loc[
    metric_wide['delta_ndcg10_semantic_minus_lexical'] < 0
].sort_values(['delta_ndcg10_semantic_minus_lexical']).head(3)
if len(semantic_wins) < 3 or len(lexical_wins) < 3:
    raise ValueError('O protocolo requer três consultas com vantagem estrita para cada método.')
comparison_query_ids = list(semantic_wins.index) + list(lexical_wins.index)
comparison_case_rows = []
comparison_top5_rows = []

for query_id in comparison_query_ids:
    query_text = str(queries[query_id].get('text', '') or '')
    grades = evaluation_judgments[query_id]
    delta = float(metric_wide.loc[query_id, 'delta_ndcg10_semantic_minus_lexical'])
    winner = 'semantic_all-MiniLM-L6-v2' if delta > 0 else 'lexical_bm25'
    case_top5 = {}
    case_evidence = {}
    for method in EVALUATION_METHODS:
        top5 = rankings_by_query[query_id][method][:5]
        top5_ids = [item['doc_id'] for item in top5]
        relevant_top5 = [doc_id for doc_id in top5_ids if grades.get(doc_id, 0.0) > 0]
        processed_query_terms = set(preprocess(query_text))
        method_matches = {}
        for rank, item in enumerate(top5, start=1):
            document = documents[item['doc_id']]
            title = str(document.get('title', '') or '')
            text = str(document.get('text', '') or '')
            processed_document_terms = set(preprocess(f'{title} {text}'))
            matched_terms = sorted(processed_query_terms & processed_document_terms)
            method_matches[item['doc_id']] = matched_terms
            comparison_top5_rows.append({
                'query_id': query_id,
                'query': query_text,
                'winner_by_ndcg10': winner,
                'delta_ndcg10_semantic_minus_lexical': delta,
                'ndcg10_lexical': float(metric_wide.loc[query_id, 'lexical_bm25']),
                'ndcg10_semantic': float(metric_wide.loc[query_id, 'semantic_all-MiniLM-L6-v2']),
                'method': method,
                'rank': rank,
                'doc_id': item['doc_id'],
                'score': float(item['score']),
                'relevance': float(grades.get(item['doc_id'], 0.0)),
                'is_relevant': grades.get(item['doc_id'], 0.0) > 0,
                'matched_bm25_query_terms': '|'.join(matched_terms),
                'matched_bm25_query_term_count': len(matched_terms),
                'title': title,
                'text': text,
            })
        case_top5[method] = top5_ids
        case_evidence[method] = {
            doc_id: method_matches[doc_id] for doc_id in top5_ids
        }
    comparison_case_rows.append({
        'query_id': query_id,
        'query': query_text,
        'winner_by_ndcg10': winner,
        'delta_ndcg10_semantic_minus_lexical': delta,
        'ndcg10_lexical': float(metric_wide.loc[query_id, 'lexical_bm25']),
        'ndcg10_semantic': float(metric_wide.loc[query_id, 'semantic_all-MiniLM-L6-v2']),
        'lexical_top5_doc_ids': json.dumps(case_top5['lexical_bm25'], ensure_ascii=False),
        'lexical_relevant_top5_doc_ids': json.dumps(
            [d for d in case_top5['lexical_bm25'] if grades.get(d, 0.0) > 0], ensure_ascii=False
        ),
        'semantic_top5_doc_ids': json.dumps(
            case_top5['semantic_all-MiniLM-L6-v2'], ensure_ascii=False
        ),
        'semantic_relevant_top5_doc_ids': json.dumps(
            [d for d in case_top5['semantic_all-MiniLM-L6-v2'] if grades.get(d, 0.0) > 0],
            ensure_ascii=False,
        ),
        'lexical_top5_query_term_matches': json.dumps(case_evidence['lexical_bm25'], ensure_ascii=False),
        'semantic_top5_query_term_matches': json.dumps(
            case_evidence['semantic_all-MiniLM-L6-v2'], ensure_ascii=False
        ),
    })

comparison_cases = pd.DataFrame(comparison_case_rows)
comparison_top5 = pd.DataFrame(comparison_top5_rows)
comparison_cases.to_csv(EXPERIMENTS_DIR / 'comparison_cases_summary.csv', index=False)
comparison_top5.to_csv(EXPERIMENTS_DIR / 'comparison_cases_top5.csv', index=False)
print('Seis casos de comparação salvos:')
print(comparison_cases[[
    'query_id', 'winner_by_ndcg10', 'delta_ndcg10_semantic_minus_lexical',
    'query',
]].to_string(index=False))
print(f"Detalhes Top5 com scores, relevância e termos coincidentes: {EXPERIMENTS_DIR / 'comparison_cases_top5.csv'}")

Seis casos de comparação salvos:
  query_id          winner_by_ndcg10  delta_ndcg10_semantic_minus_lexical                                                     query
 PLAIN-307 semantic_all-MiniLM-L6-v2                             0.671386 Vitamin D: Shedding some light on the new recommendations
PLAIN-3085 semantic_all-MiniLM-L6-v2                             0.570642  The Difficulty of Arriving at a Vitamin D Recommendation
PLAIN-1537 semantic_all-MiniLM-L6-v2                             0.567043                                            low-carb diets
PLAIN-1473              lexical_bm25                            -1.000000                                                  kohlrabi
PLAIN-2081              lexical_bm25                            -1.000000                                                shelf life
PLAIN-1288              lexical_bm25                            -0.760079                                                    grapes
Detalhes Top5 com scores, relevância e term

In [11]:
def print_comparison_top5(cases, heading):
    """Imprime os casos selecionados e os Top5 dos dois métodos com qrels."""
    print('=' * 88)
    print(heading)
    print('=' * 88)
    method_labels = {
        'lexical_bm25': 'BM25 (lexical)',
        'semantic_all-MiniLM-L6-v2': 'all-MiniLM-L6-v2 (semântico)',
    }
    for case in cases.itertuples(index=False):
        print(f"\n{case.query_id} | consulta: {case.query}")
        print(
            f"NDCG@10 BM25={case.ndcg10_lexical:.4f} | "
            f"semântico={case.ndcg10_semantic:.4f} | "
            f"Δ semântico−BM25={case.delta_ndcg10_semantic_minus_lexical:+.4f}"
        )
        for method in ('lexical_bm25', 'semantic_all-MiniLM-L6-v2'):
            print(f"  Top 5 — {method_labels[method]}:")
            docs = comparison_top5.loc[
                (comparison_top5['query_id'] == case.query_id)
                & (comparison_top5['method'] == method)
            ].sort_values('rank')
            for row in docs.itertuples(index=False):
                grade = float(row.relevance)
                relevance_label = f'RELEVANTE, grau {grade:g}' if grade > 0 else 'NÃO RELEVANTE, grau 0'
                print(
                    f"    {int(row.rank)}. [{relevance_label}] {row.doc_id} "
                    f"| score={row.score:.4f} | {row.title}"
                )

semantic_cases_to_print = comparison_cases.loc[
    comparison_cases['winner_by_ndcg10'] == 'semantic_all-MiniLM-L6-v2'
].head(3)
print_comparison_top5(
    semantic_cases_to_print,
    'TRÊS CONSULTAS COM MAIOR VANTAGEM DO MODELO SEMÂNTICO',
)

TRÊS CONSULTAS COM MAIOR VANTAGEM DO MODELO SEMÂNTICO

PLAIN-307 | consulta: Vitamin D: Shedding some light on the new recommendations
NDCG@10 BM25=0.1480 | semântico=0.8194 | Δ semântico−BM25=+0.6714
  Top 5 — BM25 (lexical):
    1. [NÃO RELEVANTE, grau 0] MED-3099 | score=15.8256 | The aryl hydrocarbon receptor and its xenobiotic ligands: a fundamental trigger for cardiovascular diseases.
    2. [NÃO RELEVANTE, grau 0] MED-4872 | score=12.8617 | Safety considerations and potential interactions of vitamins: should vitamins be considered drugs?
    3. [NÃO RELEVANTE, grau 0] MED-3373 | score=12.4809 | The Influence of Social Involvement, Neighborhood Aesthetics, and Community Garden Participation on Fruit and Vegetable Consumption
    4. [NÃO RELEVANTE, grau 0] MED-1579 | score=10.6282 | Crohn's disease: a review of treatment options and current research.
    5. [NÃO RELEVANTE, grau 0] MED-1777 | score=10.3658 | Shedding Light on the Controversy Surrounding the Temporal Decline in Huma

In [12]:
lexical_cases_to_print = comparison_cases.loc[
    comparison_cases['winner_by_ndcg10'] == 'lexical_bm25'
].head(3)
print_comparison_top5(
    lexical_cases_to_print,
    'TRÊS CONSULTAS COM MAIOR VANTAGEM DO MODELO LEXICAL (BM25)',
)

TRÊS CONSULTAS COM MAIOR VANTAGEM DO MODELO LEXICAL (BM25)

PLAIN-1473 | consulta: kohlrabi
NDCG@10 BM25=1.0000 | semântico=0.0000 | Δ semântico−BM25=-1.0000
  Top 5 — BM25 (lexical):
    1. [RELEVANTE, grau 1] MED-4455 | score=11.6606 | Glucoraphanin and 4-hydroxyglucobrassicin contents in seeds of 59 cultivars of broccoli, raab, kohlrabi, radish, cauliflower, brussels sprouts, kal...
    2. [NÃO RELEVANTE, grau 0] MED-10 | score=0.0000 | Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland
    3. [NÃO RELEVANTE, grau 0] MED-1000 | score=0.0000 | Neurobehavioral function and low-level exposure to brominated flame retardants in adolescents: a cross-sectional study
    4. [NÃO RELEVANTE, grau 0] MED-1002 | score=0.0000 | Polybrominated diphenyl ethers (PBDEs), hydroxylated PBDEs (OH-PBDEs), and measures of thyroid function in second trimester pregnant women in California
    5. [NÃO RELEVANTE, grau 0] MED-1003 | score=0.0000 | In Utero and Childhood Polybrominat